# 03 · Linear regression and Random Forest on PCA features

**Goal:** test whether the first principal components of each patient's CGM profile (`pc1`, `pc2`, `pc3`, from `02_dim_reduction.ipynb`) predict HbA1c, and compare a linear model with a Random Forest.

**Protocol:** a single train/test split is noisy with 24 test patients, so the whole pipeline is repeated over five seeds. For each seed:
1. Split the 116 patients 80/20.
2. Fit PCA on the training part only, then transform the test part (`add_pca_features`).
3. Fit linear regression, and tune a Random Forest with 5-fold CV on the training part.
4. Score both on the test part with `r2_score`.

Results are reported as mean ± standard deviation over the five seeds.

## 1. Setup

In [1]:
# Make the project root importable so `src` can be imported from this notebook
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))

In [2]:
from src.prepare_splits import load_splits, add_pca_features
from src.load_model_data import load_model_data

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

## 2. Saved splits

The saved train/test splits from `src/prepare_splits.py` already contain `pc1`, `pc2` and `pc3`, with PCA fitted on the training part. They are shown here as a check: 92 training and 24 test patients.

In [3]:
# Load the saved splits (PCA features already added)
splits = load_splits()
splits.keys()

dict_keys(['train', 'test'])

In [4]:
# Check split sizes and columns
for split_name, data in splits.items():
    print(f'Split: {split_name}, n_samples: {len(data)}')
    print(data.columns)

Split: train, n_samples: 92
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion', 'pc1', 'pc2', 'pc3'],
      dtype='str')
Split: test, n_samples: 24
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion', 'pc1', 'pc2', 'pc3'],
      dtype='str')


## 3. Seed experiment

**Why not the saved splits?** The saved split is one random 80/20 split. Repeating the split over several seeds shows how much the results depend on which patients land in the test set. The test sets overlap across seeds, so the standard deviation understates the real uncertainty.

In [5]:
# Shared settings for the seed experiment
SEEDS = [0, 1, 2, 3, 4]
FEATURES = ['pc1', 'pc2', 'pc3']
TARGET = 'hba1c'

model_data = load_model_data()

rf_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'max_features': ['sqrt', 1.0],
}

In [ ]:
# Each seed gives a new 80/20 split; PCA is fitted on its training part only
results = []
for seed in SEEDS:
    train_raw, test_raw = train_test_split(model_data, test_size=0.2, random_state=seed)
    train_df, test_df = add_pca_features(train_raw.reset_index(drop=True), test_raw.reset_index(drop=True))

    X_train, y_train = train_df[FEATURES], train_df[TARGET]
    X_test, y_test = test_df[FEATURES], test_df[TARGET]

    lr = LinearRegression().fit(X_train, y_train)

    rf_search = GridSearchCV(
        RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1),
        rf_grid, cv=5, scoring='r2'
    ).fit(X_train, y_train)
    rf = rf_search.best_estimator_

    for name, model in [('linear', lr), ('random_forest', rf)]:
        pred = model.predict(X_test)
        results.append({
            'seed': seed,
            'model': name,
            'r2': r2_score(y_test, pred),
            'rmse': np.sqrt(mean_squared_error(y_test, pred)),
            'mae': mean_absolute_error(y_test, pred),
            'best_params': rf_search.best_params_ if name == 'random_forest' else None,
        })

results_df = pd.DataFrame(results)
print(results_df.round(3))
print(results_df.groupby('model')[['r2', 'rmse', 'mae']].agg(['mean', 'std']).round(3))

## 4. Baseline: mean-only predictor

**Goal:** a floor for comparison. This model ignores all features and predicts the training mean HbA1c for every test patient. Any useful model has to beat it.

On the training data its R² is 0 by construction. On new data it is slightly negative, because the test mean differs from the training mean.

In [ ]:
# Mean-only predictor on the same five seed splits as the models above
from sklearn.dummy import DummyRegressor

mean_rows = []
for seed in SEEDS:
    train_raw, test_raw = train_test_split(model_data, test_size=0.2, random_state=seed)
    dummy = DummyRegressor(strategy='mean').fit(train_raw, train_raw[TARGET])
    pred = dummy.predict(test_raw)
    y_test = test_raw[TARGET]
    mean_rows.append({
        'seed': seed,
        'model': 'mean_only',
        'r2': r2_score(y_test, pred),
        'rmse': np.sqrt(mean_squared_error(y_test, pred)),
        'mae': mean_absolute_error(y_test, pred),
        'best_params': None,
    })

mean_df = pd.DataFrame(mean_rows)
print(mean_df.round(3))
print(pd.concat([results_df, mean_df])
      .groupby('model')[['r2', 'rmse', 'mae']].agg(['mean', 'std']).round(3))

   seed      model     r2    rmse     mae best_params
0     0  mean_only -0.024  32.184  25.474        None
1     1  mean_only -0.170  29.496  26.125        None
2     2  mean_only -0.134  31.301  23.363        None
3     3  mean_only -0.030  31.773  27.798        None
4     4  mean_only -0.002  28.492  21.778        None
                  r2           rmse            mae       
                mean    std    mean    std    mean    std
model                                                    
linear         0.235  0.131  25.903  3.351  19.857  2.019
mean_only     -0.072  0.075  30.649  1.583  24.908  2.363
random_forest  0.134  0.136  27.551  3.138  21.296  1.408


## 5. Summary

| Model | Mean R² | Std R² | Mean RMSE | Mean MAE |
|---|---|---|---|---|
| Mean-only predictor | -0.072 | 0.075 | 30.65 | 24.91 |
| Linear regression | 0.235 | 0.131 | 25.90 | 19.86 |
| Random Forest (tuned) | 0.134 | 0.136 | 27.55 | 21.30 |

### Observations
- **Mean-only predictor (R² -0.07 on average):** linear regression beats it on all five seeds. The Random Forest beats it on four of five; on seed 0 it is slightly below the mean-only predictor (-0.04 against -0.02).
- **Linear regression beats the Random Forest on every seed**, with R² between 0.05 and 0.35 against -0.04 to 0.31 for the forest.
- **The forest's tuned settings change with the seed**: the best `max_depth` alternates between 3 and 5, and the best `max_features` between `sqrt` and `1.0`. The grid search is picking up noise.
- **The spread across seeds is as large as the gap between models.** The R² standard deviation is about 0.13 for both models.
- The three PCA components carry only a small amount of information, so there is little for a Random Forest to exploit beyond what a linear model already captures.

### Conclusions
- **With PCA features, plain linear regression is the better model**, and the Random Forest does not add value.
- **The gain is modest and uncertain.** A mean R² of about 0.24 across seeds is positive but not large, and HbA1c is still only partly explained by the CGM profile.
- These numbers are not directly comparable with the single-split results in `01_linear_model.ipynb`, because the protocol differs. A like-for-like comparison would use the same seed protocol with the TAR/TBR features.

### Next steps
- Vary the number of PCA components, for example 2 to 5, and check the effect on the seed results.